# Sales Prediction using Machine Learning in Python

# Import Libraries

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML Libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Model Saving
import joblib
import os

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")


# Load Dataset

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
df = pd.read_csv("/kaggle/input/advertising-csv/advertising (1).csv")
df.head()


In [ ]:
df.info()
df.describe()


# Exploratory Data Analysis (EDA)

# Correlation Matrix

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()


# Pairplot

In [ ]:
sns.pairplot(df)
plt.show()


# Distribution Plots

In [ ]:
df.hist(figsize=(10,8))
plt.show()


# Feature vs Target Relationship

In [ ]:
features = ['TV','Radio','Newspaper']

for feature in features:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df[feature], y=df['Sales'])
    plt.title(f"{feature} vs Sales")
    plt.show()


# Target Variable Deep Analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Sales'], kde=True, bins=20)
plt.title("Sales Distribution with KDE")
plt.show()

print("Skewness:", df['Sales'].skew())
print("Kurtosis:", df['Sales'].kurt())


# Outlier Detection (Boxplots)

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=df)
plt.title("Boxplot for Outlier Detection")
plt.show()


# Using IQR Method

In [ ]:
def detect_outliers(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers

for col in df.columns:
    print(f"{col} Outliers:", len(detect_outliers(col)))


# Multicollinearity Check

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = df.drop("Sales", axis=1)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(len(X_vif.columns))
]

vif_data


# Feature Interaction Analysis

In [ ]:
sns.lmplot(x="TV", y="Sales", hue="Radio", data=df)
plt.title("TV vs Sales with Radio Interaction")
plt.show()


# Advertising Budget Contribution %

In [ ]:
df['Total_Ad_Spend'] = df['TV'] + df['Radio'] + df['Newspaper']

df['TV_ratio'] = df['TV'] / df['Total_Ad_Spend']
df['Radio_ratio'] = df['Radio'] / df['Total_Ad_Spend']
df['Newspaper_ratio'] = df['Newspaper'] / df['Total_Ad_Spend']

df[['TV_ratio','Radio_ratio','Newspaper_ratio']].head()


# Sales vs Total Advertising Spend

In [ ]:
plt.figure(figsize=(7,5))
sns.regplot(x=df['Total_Ad_Spend'], y=df['Sales'])
plt.title("Total Advertising Spend vs Sales")
plt.show()


# Residual Analysis (After Linear Regression)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])




In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("Sales", axis=1)
y = df["Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
linear_pipeline.fit(X_train, y_train)
y_pred = linear_pipeline.predict(X_test)

residuals = y_test - y_pred

plt.figure(figsize=(6,4))
sns.scatterplot(x=y_pred, y=residuals)
plt.axhline(0, color='red')
plt.title("Residual Plot")
plt.xlabel("Predicted Sales")
plt.ylabel("Residuals")
plt.show()


# Normality Test (Shapiro-Wilk)

In [ ]:
from scipy.stats import shapiro

stat, p = shapiro(df['Sales'])
print("p-value:", p)

if p > 0.05:
    print("Sales looks normally distributed")
else:
    print("Sales is not normally distributed")


# Correlation with P-values (Statistical Significance)

In [ ]:
import scipy.stats as stats

for col in ['TV','Radio','Newspaper']:
    corr, p_value = stats.pearsonr(df[col], df['Sales'])
    print(f"{col} Correlation: {corr:.3f}, P-value: {p_value:.5f}")


# Polynomial Relationship Check

In [ ]:
sns.regplot(x="TV", y="Sales", data=df, order=2)
plt.title("Polynomial Trend (TV vs Sales)")
plt.show()



# Advanced Heatmap with Mask

In [ ]:
plt.figure(figsize=(8,6))
corr_matrix = df.corr()

mask = np.triu(corr_matrix)

sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", mask=mask)
plt.title("Advanced Correlation Heatmap")
plt.show()


# Clustering Visualization

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(df[['TV','Radio','Newspaper']])

sns.scatterplot(x="TV", y="Sales", hue="Cluster", data=df)
plt.title("Clustered Advertising Strategy")
plt.show()


# Feature Engineering

In [ ]:
X = df.drop("Sales", axis=1)
y = df["Sales"]

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Model Building (Multiple Algorithms)

# Linear Regression Pipeline

In [ ]:
linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_pipeline.fit(X_train, y_train)
y_pred_lr = linear_pipeline.predict(X_test)

print("Linear Regression R2:", r2_score(y_test, y_pred_lr))


# Random Forest (Advanced Model)

In [ ]:
rf_pipeline = Pipeline([
    ("model", RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("Random Forest R2:", r2_score(y_test, y_pred_rf))


# Gradient Boosting

In [ ]:
gb_pipeline = Pipeline([
    ("model", GradientBoostingRegressor(random_state=42))
])

gb_pipeline.fit(X_train, y_train)
y_pred_gb = gb_pipeline.predict(X_test)

print("Gradient Boosting R2:", r2_score(y_test, y_pred_gb))


# Model Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print("R2 Score:", r2)
    print("MAE:", mae)
    print("RMSE:", rmse)
    
    return r2


In [ ]:
print("Linear Regression")
evaluate_model(linear_pipeline, X_test, y_test)

print("\nRandom Forest")
evaluate_model(rf_pipeline, X_test, y_test)

print("\nGradient Boosting")
evaluate_model(gb_pipeline, X_test, y_test)


# Hyperparameter Tuning

In [ ]:
param_grid = {
    "model__n_estimators": [100,200,300],
    "model__max_depth": [None,5,10]
}

grid = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)


# Feature Importance

# Save Best Model 

In [ ]:

best_model = grid.best_estimator_


In [ ]:
best_model = rf_pipeline   # or linear_pipeline or gb_pipeline


In [ ]:
import os
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/best_model.pkl")
print("Model Saved Successfully!")


In [ ]:
import os
import joblib

MODEL_DIR = "models"
MODEL_PATH = os.path.join(MODEL_DIR, "best_model.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)

try:
    joblib.dump(best_model, MODEL_PATH)
    print(f"Model saved successfully at {MODEL_PATH}")
except Exception as e:
    print("Error saving model:", e)


# Prediction Function (Deployment Ready)

In [ ]:

def predict_sales(tv, radio, newspaper):
    import joblib
    import pandas as pd
    
    model = joblib.load("models/best_model.pkl")
    
    # Create base dataframe
    input_data = pd.DataFrame({
        "TV": [tv],
        "Radio": [radio],
        "Newspaper": [newspaper]
    })
    
    # ---- Feature Engineering (MUST match training) ----
    input_data["Total_Ad_Spend"] = (
        input_data["TV"] +
        input_data["Radio"] +
        input_data["Newspaper"]
    )
    
    input_data["TV_ratio"] = input_data["TV"] / input_data["Total_Ad_Spend"]
    input_data["Radio_ratio"] = input_data["Radio"] / input_data["Total_Ad_Spend"]
    input_data["Newspaper_ratio"] = input_data["Newspaper"] / input_data["Total_Ad_Spend"]
    
    # If clustering was used
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=3, random_state=42)
    kmeans.fit(df[['TV','Radio','Newspaper']])   # fit on training structure
    input_data["Cluster"] = kmeans.predict(input_data[['TV','Radio','Newspaper']])
    
    # -----------------------------------------------
    
    prediction = model.predict(input_data)[0]
    
    return round(prediction, 2)


In [ ]:
from sklearn.preprocessing import FunctionTransformer

def feature_engineering(X):
    X = X.copy()
    X["Total_Ad_Spend"] = X["TV"] + X["Radio"] + X["Newspaper"]
    X["TV_ratio"] = X["TV"] / X["Total_Ad_Spend"]
    X["Radio_ratio"] = X["Radio"] / X["Total_Ad_Spend"]
    X["Newspaper_ratio"] = X["Newspaper"] / X["Total_Ad_Spend"]
    return X

feature_transformer = FunctionTransformer(feature_engineering)

pipeline = Pipeline([
    ("feature_engineering", feature_transformer),
    ("model", RandomForestRegressor())
])
